---
title: Examples
description: Practical examples of using the Executor primitive in Qiskit Runtime.
---


# Executor examples

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

The examples in this section illustrate some common ways to use the Executor primitive. Before running these examples, follow the instructions in [Install Qiskit](/docs/guides/install-qiskit) and [Get started with Executor](/docs/guides/directed-execution-model).

## Example: Parameterized circuit

This example illustrates how to add circuit items with parameters, as well as how to add samplex items.

The examples in this section use the following circuit, which generates a three-qubit GHZ state, rotates
the qubits around the Pauli-Z axis, and measures the qubits in the computational basis. We show how
to add this circuit to a `QuantumProgram`, optionally randomizing its content with twirling
gates, and execute the program by using the  `Executor` class.

In [2]:
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Generate the circuit
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)
circuit.h(2)
circuit.cz(1, 2)
circuit.h(2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)
circuit.rz(Parameter("lam"), 2)
circuit.measure_all()

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### Define a `QuantumProgram`

The input to Executor is a `QuantumProgram`. For full details, see [Executor input and output](/docs/guides/executor-input-output#programs). Note that Samplomatic uses [`generate_boxing_pass_manager`](https://qiskit.github.io/samplomatic/guides/transpiler.html) for transpilation.

In [3]:
# Initialize an empty program
program = QuantumProgram(shots=1024)


# Append the circuit and the parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(10, 3),  # 10 sets of parameter values
)

# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template and the samplex
template, samplex = build(boxed_circuit)

# Append the template and samplex as a samplex item
program.append_samplex_item(
    template,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 3),
    },
    shape=(2, 14, 10),
)

### Run the Executor job

In [12]:
# initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Example: Perform PEC

This example shows how to use a samplex item to perform probabilistic error cancellation ([PEC](/docs/guides/error-mitigation-and-suppression-techniques#pec)) for error mitigation.

Consider a mirrored-version of a circuit with ten qubits and two unique layers of CX gates. These are the main tasks:
- Execute the circuit with twirling.
- Execute the circuit with PEC mitigation, as in the paper ["Probabilistic error cancellation with sparse Pauli-Lindblad models on noisy quantum processors"](https://arxiv.org/abs/2201.09866).

The pipeline consists of these steps:
1. Setup: Generate the target circuit and group its operations into boxes.
2. Learn: Learn the noise of the instructions that we want to mitigate with PEC.
3. Execute: Run the circuit on a backend and analyze the results.

### Set up the circuit

Choose a backend and prepare a 10-qubit circuit.

In [5]:
from qiskit_ibm_runtime import QiskitRuntimeService

token = "<YOUR_TOKEN>"
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)

# Prepare a circuit

num_qubits = 10
num_layers = 10

qubits = list(range(num_qubits))
circuit = QuantumCircuit(num_qubits)

for layer_idx in range(num_layers):
    circuit.rx(Parameter(f"theta_{layer_idx}"), qubits)
    for i in range(num_qubits // 2):
        circuit.cz(qubits[2 * i], qubits[2 * i + 1])

    circuit.rx(Parameter(f"phi_{layer_idx}"), qubits)
    for i in range(num_qubits // 2 - 1):
        circuit.cz(qubits[2 * i] + 1, qubits[2 * i + 1] + 1)

circuit.draw("mpl", scale=0.35, fold=100)

<Image src="/docs/images/guides/executor-examples/extracted-outputs/72b374ae-66b2-40d4-ae06-0c8ceec60c36-0.avif" alt="Output of the previous code cell" />

Combine the circuit with its inverse to create a mirror circuit.

In [6]:
mirror_circuit = circuit.compose(circuit.inverse())
mirror_circuit.measure_all()

mirror_circuit.draw("mpl", scale=0.35, fold=100)

<Image src="/docs/images/guides/executor-examples/extracted-outputs/b63a0d25-88c3-4795-b2e4-a2a5157b9f52-0.avif" alt="Output of the previous code cell" />

Set some parameter values:

In [7]:
import numpy as np

parameter_values = np.random.rand(mirror_circuit.num_parameters)

Use the pass manager to transpile the circuit against the backend, and group gates and measurements into annotated boxes. Begin by creating a circuit with twirled-annotated boxes.

In [8]:
preset_pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(range(40, 50)),  # qubits 40 to 50
    optimization_level=0,
)

# Run the boxing pass manager after the scheduling stage
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)

mirror_circuit_twirl = preset_pass_manager.run(mirror_circuit)

Next, generate a new boxed circuit with `Twirl` and `InjectNoise` annotations.

In [9]:
preset_pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(range(40, 50)),
    optimization_level=0,
)

# Run the boxing pass manager after the scheduling stage
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)

mirror_circuit_pec = preset_pass_manager.run(mirror_circuit)

### Learn the noise

To minimize the number of noise learning experiments, identify the unique instructions in the second circuit (the one with boxes annotated with `InjectNoise`). In defining uniqueness, two box instructions are equal if both of the following are true:
- Their content is equal, up to single-qubit gates.
- Their `Twirl` annotation is equal (every other annotation is disregarded).

This leads to three unique instructions, namely the odd and even gate boxes, and the final measurement box.

In [10]:
from samplomatic.utils import find_unique_box_instructions

unique_box_instructions = find_unique_box_instructions(
    mirror_circuit_pec.data
)
assert len(unique_box_instructions) == 3

Initialize a `NoiseLearnerV3`, choose the learning parameters by setting its options, and run a noise learning job.

In [13]:
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

learner = NoiseLearnerV3(backend)

learner.options.shots_per_randomization = 128
learner.options.num_randomizations = 32
learner.options.layer_pair_depths = [0, 1, 2, 4, 16, 32]

learner_job = learner.run(unique_box_instructions)

learner_job.job_id()
learner_result = learner_job.result()

Convert `result` to the object required by the samplex by using the `result.to_dict` method.

In [14]:
noise_maps = learner_result.to_dict(
    instructions=unique_box_instructions, require_refs=False
)

### Execute the circuits

`Executor` runs `QuantumProgram` objects. Each `QuantumProgram` can contain several *items*, which can be thought of as a (template, samplex) pair.

Initialize an empty program, requesting `1000` shots for each of its items.

In [15]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram

# Generate a quantum program
program = QuantumProgram(shots=1000)

Next, append the template and samplex built for `mirror_circuit_twirl`, requesting `900` randomizations. This means that the samplex will produce `900` sets of parameters, and each set will be executed `1000` times (the number of shots) in the QPU.

In [16]:
template_twirl, samplex_twirl = build(mirror_circuit_twirl)

program.append_samplex_item(
    template_twirl,
    samplex=samplex_twirl,
    samplex_arguments={"parameter_values": parameter_values},
    shape=(900,),
)

Similarly, append the template and samplex built for `mirror_circuit_pec`, requesting `900` randomizations.

In [17]:
template_pec, samplex_pec = build(mirror_circuit_pec)

program.append_samplex_item(
    template_pec,
    samplex=samplex_pec,
    samplex_arguments={
        "parameter_values": parameter_values,
        "pauli_lindblad_maps": noise_maps,
        "noise_scales": {
            ref: -1.0 for ref in noise_maps
        },  # Set the scales to -1 for PEC
    },
    shape=(900,),
)

Import `Executor` and submit a job.

In [19]:
from qiskit_ibm_runtime.executor import Executor

executor = Executor(backend)
executor_job = executor.run(program)

executor_job.job_id()

executor_results = executor_job.result()
executor_results

twirl_result = executor_results[0]

print(f"Twirl result keys:\n {list(twirl_result.keys())}\n")
print(f"Shape of results: {twirl_result['meas'].shape}")

pec_result = executor_results[1]

print(f"PEC result keys:\n {list(pec_result.keys())}\n")
print(f"Shape of results: {pec_result['meas'].shape}")

Twirl result keys:
 ['meas', 'measurement_flips.meas']

Shape of results: (900, 1000, 10)
PEC result keys:
 ['meas', 'measurement_flips.meas', 'pauli_signs']

Shape of results: (900, 1000, 10)


### Analyze results

Finally, post-process the results to estimate the expectation values of single-qubit Pauli-Z operators acting on each of the ten active qubits (expected value: `1.0`).

In [20]:
# Undo measurement twirling
twirl_result_unflipped = (
    twirl_result["meas"] ^ twirl_result["measurement_flips.meas"]
)

# Calculate the expectation values of single-qubit Z operators
exp_vals = 1 - 2 * twirl_result_unflipped.mean(axis=1).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 0.36
Qubit 1 -> 0.26
Qubit 2 -> 0.2
Qubit 3 -> 0.12
Qubit 4 -> 0.26
Qubit 5 -> 0.31
Qubit 6 -> 0.39
Qubit 7 -> 0.45
Qubit 8 -> 0.59
Qubit 9 -> 0.61


In [21]:
# Undo measurement twirling
pec_result_unflipped = (
    pec_result["meas"] ^ pec_result["measurement_flips.meas"]
)

# Calculate the signs for PEC mitigation
signs = np.prod((-1) ** pec_result["pauli_signs"], axis=-1)
signs = signs.reshape((signs.shape[0], 1))

# Calculate the expectation values of single-qubit Z operators as required by
# PEC mitigation
exp_vals = 1 - (2 * pec_result_unflipped.mean(axis=1) * signs).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 1.02
Qubit 1 -> 1.01
Qubit 2 -> 1.03
Qubit 3 -> 1.03
Qubit 4 -> 1.03
Qubit 5 -> 1.03
Qubit 6 -> 1.03
Qubit 7 -> 1.03
Qubit 8 -> 1.03
Qubit 9 -> 1.03


## Next steps

<Admonition type="tip" title="Recommendations">
    - Review the [broadcasting](/docs/guides/primitive-input-output#broadcasting) overview.
    - Learn how to use [Executor options](/docs/guides/executor-options).
    - Understand the [directed execution model](/docs/guides/directed-execution-model).
    - Review the [Samplomatic documentation](https://qiskit.github.io/samplomatic/).
</Admonition>